# AlphaMoo Phase 0 — Real LLM Measurement on Kaggle RTX 6000

This notebook measures **actual** LLM inference latency on the Kaggle RTX 6000 GPU, replacing the latency-model estimates from `scripts/phase0_runner.py`.

## What this measures

For each model variant:
1. Cold-start load time (model + tokenizer + KV cache setup)
2. Warm inference: 100 calls with our actual Phase 0 prompt
3. Per-call wall-clock (prefill + decode)
4. Tokens/sec (prefill and decode separately)
5. Peak GPU memory

## Output

A `real_compute_profile.json` file that supersedes `docs/phase0_compute_profile.md`.

## How to run

1. Set accelerator to **GPU RTX 6000** in Kaggle notebook settings
2. Set internet to **OFF** (matches competition conditions) — but you'll need internet ON for the first run to download model weights. After that, cache them and turn internet off.
3. Run all cells top-to-bottom
4. The final cell prints the real ComputeProfile and saves it as JSON

## Step 1: Install dependencies

If internet is off, you'll need to add these as Kaggle datasets instead. For first run, internet on is OK.

In [ ]:
# Install vLLM and transformers (requires internet on first run)
!pip install -q vllm transformers accelerate bitsandbytes
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

## Step 2: Verify GPU is available

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError("No GPU available! Set accelerator to RTX 6000 in Kaggle settings.")

## Step 3: Define our actual Phase 0 prompt

This is the exact prompt the agent will use in production. Measuring with this prompt gives us real numbers, not synthetic.

In [ ]:
PHASE0_PROMPT = """# ARC-AGI-3 Agent — Game ls20
Level: 0/7

## Available Actions
1, 2, 3, 4

## Recent Action History (last 10)
(none)

## Current Scene
Background color: 4
Objects detected: 16
  - obj_001: color=4, cells=2609, topology=solid, bbox=(0, 0, 63, 63)
  - obj_002: color=3, cells=892, topology=hollow, bbox=(9, 5, 58, 54)
  - obj_003: color=5, cells=208, topology=solid, bbox=(0, 0, 3, 51)
  - obj_004: color=11, cells=84, topology=solid, bbox=(13, 61, 54, 62)
  - obj_005: color=9, cells=45, topology=solid, bbox=(20, 50, 30, 53)
  - obj_006: color=12, cells=10, topology=solid, bbox=(35, 50, 39, 53)
  - obj_007: color=0, cells=3, topology=solid, bbox=(22, 30, 24, 31)
  - obj_008: color=1, cells=2, topology=solid, bbox=(21, 30, 22, 31)

## Agent State
Position: (21, 30)
Colors: [0, 1]
Shape hash: a3f2b9c1d8e7f4a2

## Task
Select the next action. Respond with one action ID and brief reasoning.

Action:"""

print(f"Prompt length: {len(PHASE0_PROMPT)} chars (~{len(PHASE0_PROMPT)//4} tokens)")

## Step 4: Define the measurement function

Loads a model, runs 100 warm inference calls, returns real stats.

In [ ]:
import time
import json
import statistics
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class RealMeasurement:
    model_name: str
    quantization: str
    cold_load_sec: float
    peak_vram_gb: float
    n_warmup_calls: int
    n_measured_calls: int
    prompt_tokens: int
    max_output_tokens: int
    prefill_tokens_per_sec: float
    decode_tokens_per_sec: float
    avg_call_ms: float
    p50_call_ms: float
    p95_call_ms: float
    p99_call_ms: float
    max_call_ms: float
    total_wall_sec: float

def measure_model(
    model_name: str,
    quantization: str = "awq",
    n_calls: int = 100,
    n_warmup: int = 5,
    max_output_tokens: int = 80,
) -> Optional[RealMeasurement]:
    """Measure a model's real inference latency on this GPU."""
    print(f"\n{'='*60}")
    print(f"Measuring: {model_name} ({quantization})")
    print(f"{'='*60}")
    
    try:
        from vllm import LLM, SamplingParams
    except ImportError:
        print("vLLM not available, trying transformers + bitsandbytes...")
        return measure_with_transformers(model_name, quantization, n_calls, n_warmup, max_output_tokens)
    
    # Cold load
    t0 = time.perf_counter()
    try:
        llm = LLM(
            model=model_name,
            quantization=quantization if quantization != "none" else None,
            max_model_len=4096,
            gpu_memory_utilization=0.85,
            enforce_eager=False,  # enable CUDA graphs
            enable_prefix_caching=True,
        )
        cold_load_sec = time.perf_counter() - t0
        print(f"Cold load: {cold_load_sec:.1f}s")
    except Exception as e:
        print(f"Failed to load with vLLM: {e}")
        return None
    
    # Peak VRAM
    torch.cuda.synchronize()
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak VRAM after load: {peak_vram_gb:.2f} GB")
    
    # Tokenize prompt to count tokens
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    prompt_tokens = len(tokenizer.encode(PHASE0_PROMPT))
    print(f"Prompt tokens: {prompt_tokens}")
    
    # Warmup
    print(f"Warming up ({n_warmup} calls)...")
    sampling = SamplingParams(
        temperature=0.0,
        max_tokens=max_output_tokens,
        stop=["\n\n"]
    )
    for _ in range(n_warmup):
        llm.generate([PHASE0_PROMPT], sampling)
    
    # Reset peak memory for measurement
    torch.cuda.reset_peak_memory_stats()
    
    # Measure
    print(f"Measuring ({n_calls} calls)...")
    call_times_ms = []
    total_output_tokens = 0
    
    t_start = time.perf_counter()
    for i in range(n_calls):
        t0 = time.perf_counter()
        outputs = llm.generate([PHASE0_PROMPT], sampling)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        call_times_ms.append((t1 - t0) * 1000)
        # Count output tokens
        out_tokens = len(outputs[0].outputs[0].token_ids)
        total_output_tokens += out_tokens
        if (i + 1) % 20 == 0:
            print(f"  call {i+1}/{n_calls}: {call_times_ms[-1]:.1f}ms, {out_tokens} output tokens")
    total_wall_sec = time.perf_counter() - t_start
    
    # Compute stats
    avg_call_ms = statistics.mean(call_times_ms)
    sorted_times = sorted(call_times_ms)
    n = len(sorted_times)
    p50 = sorted_times[n // 2]
    p95 = sorted_times[int(n * 0.95)]
    p99 = sorted_times[min(int(n * 0.99), n - 1)]
    max_call = sorted_times[-1]
    
    # Throughput
    avg_output_per_call = total_output_tokens / n_calls
    decode_tokens_per_sec = avg_output_per_call / (avg_call_ms / 1000)
    prefill_tokens_per_sec = prompt_tokens / (avg_call_ms / 1000)  # rough
    
    # Peak VRAM during measurement
    peak_vram_during = torch.cuda.max_memory_allocated() / 1024**3
    
    result = RealMeasurement(
        model_name=model_name,
        quantization=quantization,
        cold_load_sec=cold_load_sec,
        peak_vram_gb=peak_vram_during,
        n_warmup_calls=n_warmup,
        n_measured_calls=n_calls,
        prompt_tokens=prompt_tokens,
        max_output_tokens=max_output_tokens,
        prefill_tokens_per_sec=prefill_tokens_per_sec,
        decode_tokens_per_sec=decode_tokens_per_sec,
        avg_call_ms=avg_call_ms,
        p50_call_ms=p50,
        p95_call_ms=p95,
        p99_call_ms=p99,
        max_call_ms=max_call,
        total_wall_sec=total_wall_sec,
    )
    
    print(f"\nResults for {model_name}:")
    print(f"  Avg call: {avg_call_ms:.1f}ms")
    print(f"  P50/P95/P99: {p50:.1f}/{p95:.1f}/{p99:.1f}ms")
    print(f"  Decode throughput: {decode_tokens_per_sec:.1f} tok/s")
    print(f"  Peak VRAM: {peak_vram_during:.2f} GB")
    
    # Clean up
    del llm
    torch.cuda.empty_cache()
    
    return result


def measure_with_transformers(
    model_name: str,
    quantization: str,
    n_calls: int,
    n_warmup: int,
    max_output_tokens: int,
) -> Optional[RealMeasurement]:
    """Fallback: use transformers + bitsandbytes if vLLM fails."""
    print(f"Using transformers fallback for {model_name}")
    
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch
    
    # Cold load
    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if quantization == "4bit":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
    elif quantization == "awq":
        # AWQ models load via transformers directly
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map="auto", torch_dtype=torch.float16
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map="auto", torch_dtype=torch.float16
        )
    
    cold_load_sec = time.perf_counter() - t0
    print(f"Cold load: {cold_load_sec:.1f}s")
    
    torch.cuda.synchronize()
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
    
    prompt_tokens = len(tokenizer.encode(PHASE0_PROMPT))
    input_ids = tokenizer.encode(PHASE0_PROMPT, return_tensors="pt").cuda()
    
    # Warmup
    for _ in range(n_warmup):
        _ = model.generate(input_ids, max_new_tokens=max_output_tokens, do_sample=False)
    
    torch.cuda.reset_peak_memory_stats()
    
    call_times_ms = []
    total_output_tokens = 0
    
    t_start = time.perf_counter()
    for i in range(n_calls):
        t0 = time.perf_counter()
        outputs = model.generate(input_ids, max_new_tokens=max_output_tokens, do_sample=False)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        call_times_ms.append((t1 - t0) * 1000)
        out_tokens = outputs.shape[1] - input_ids.shape[1]
        total_output_tokens += out_tokens
    total_wall_sec = time.perf_counter() - t_start
    
    avg_call_ms = statistics.mean(call_times_ms)
    sorted_times = sorted(call_times_ms)
    n = len(sorted_times)
    
    avg_output_per_call = total_output_tokens / n_calls
    decode_tokens_per_sec = avg_output_per_call / (avg_call_ms / 1000)
    
    peak_vram_during = torch.cuda.max_memory_allocated() / 1024**3
    
    result = RealMeasurement(
        model_name=model_name,
        quantization=quantization + " (transformers)",
        cold_load_sec=cold_load_sec,
        peak_vram_gb=peak_vram_during,
        n_warmup_calls=n_warmup,
        n_measured_calls=n_calls,
        prompt_tokens=prompt_tokens,
        max_output_tokens=max_output_tokens,
        prefill_tokens_per_sec=prompt_tokens / (avg_call_ms / 1000),
        decode_tokens_per_sec=decode_tokens_per_sec,
        avg_call_ms=avg_call_ms,
        p50_call_ms=sorted_times[n // 2],
        p95_call_ms=sorted_times[int(n * 0.95)],
        p99_call_ms=sorted_times[min(int(n * 0.99), n - 1)],
        max_call_ms=sorted_times[-1],
        total_wall_sec=total_wall_sec,
    )
    
    del model
    torch.cuda.empty_cache()
    return result

## Step 5: Run measurements on candidate models

We'll test the same 3 models from the Phase 0 stub:
- Qwen2.5-0.5B-Instruct (4-bit AWQ)
- Qwen2.5-1.5B-Instruct (4-bit AWQ)
- VibeThinker-3B (4-bit, if it loads)

AWQ quantized variants are pre-quantized by the community. We use them directly.

In [ ]:
# Candidate models — using community AWQ quants from HuggingFace
CANDIDATES = [
    # (model_id, quantization_label, n_calls)
    ("Qwen/Qwen2.5-0.5B-Instruct-AWQ", "awq", 100),
    ("Qwen/Qwen2.5-1.5B-Instruct-AWQ", "awq", 100),
    # VibeThinker doesn't have an official AWQ; try BnB 4-bit fallback
    # ("WeiboAI/VibeThinker-3B", "4bit", 50),  # uncomment to test
]

all_results = []
for model_name, quant, n_calls in CANDIDATES:
    result = measure_model(model_name, quant, n_calls=n_calls)
    if result is not None:
        all_results.append(result)
    else:
        print(f"Skipped {model_name} (failed to load)")

## Step 6: Compute the real ComputeProfile

For each measured model, project to the 14,800-action hidden set and check against the 9-hour budget.

In [ ]:
KAGGLE_BUDGET_HOURS = 9
KAGGLE_BUDGET_SEC = KAGGLE_BUDGET_HOURS * 3600
ESTIMATED_HIDDEN_ACTIONS = 14800

print(f"\n{'='*80}")
print(f"REAL Phase 0 ComputeProfile (measured on {torch.cuda.get_device_name(0)})")
print(f"{'='*80}")
print(f"Kaggle budget: {KAGGLE_BUDGET_HOURS} hours ({KAGGLE_BUDGET_SEC:,} sec)")
print(f"Estimated hidden set: {ESTIMATED_HIDDEN_ACTIONS:,} actions")
print()

print(f"{'Model':<40} {'Avg ms':>10} {'P95 ms':>10} {'Proj (h)':>10} {'Fits?':>8} {'Headroom':>10}")
print("-" * 90)

for r in all_results:
    projected_sec = r.avg_call_ms * ESTIMATED_HIDDEN_ACTIONS / 1000
    projected_hr = projected_sec / 3600
    fits = projected_sec <= KAGGLE_BUDGET_SEC
    headroom = (1 - projected_sec / KAGGLE_BUDGET_SEC) * 100 if fits else 0
    fits_str = "YES" if fits else "NO"
    headroom_str = f"{headroom:.1f}%" if fits else "OVER"
    
    print(f"{r.model_name[:38]:<40} {r.avg_call_ms:>10.1f} {r.p95_call_ms:>10.1f} "
          f"{projected_hr:>10.2f} {fits_str:>8} {headroom_str:>10}")

print()

## Step 7: Save the real ComputeProfile as JSON

This replaces the stub-based estimate in `docs/phase0_compute_profile.md`.

In [ ]:
output = {
    "measurement_date": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "kaggle_budget_hours": KAGGLE_BUDGET_HOURS,
    "estimated_hidden_actions": ESTIMATED_HIDDEN_ACTIONS,
    "prompt": PHASE0_PROMPT,
    "measurements": [asdict(r) for r in all_results],
}

with open("real_compute_profile.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved real_compute_profile.json")
print(f"\nFile size: {len(json.dumps(output))} bytes")
print("\nFirst measurement summary:")
print(json.dumps(output["measurements"][0], indent=2))

## Step 8: Compare real vs stub estimate

This tells us how accurate the stub latency model was.

In [ ]:
# Stub estimates from src/alphamoo/llm_stub.py
STUB_ESTIMATES = {
    "Qwen/Qwen2.5-0.5B-Instruct-AWQ": 1467.8,  # ms/step from Phase 0 report
    "Qwen/Qwen2.5-1.5B-Instruct-AWQ": 2691.3,
}

print(f"\n{'='*60}")
print("Stub vs Real comparison")
print(f"{'='*60}")
print(f"{'Model':<40} {'Stub ms':>10} {'Real ms':>10} {'Diff %':>10}")
print("-" * 70)

for r in all_results:
    stub = STUB_ESTIMATES.get(r.model_name)
    if stub:
        diff_pct = (r.avg_call_ms - stub) / stub * 100
        sign = "+" if diff_pct > 0 else ""
        print(f"{r.model_name[:38]:<40} {stub:>10.1f} {r.avg_call_ms:>10.1f} "
              f"{sign}{diff_pct:>9.1f}%")

print("\nNegative diff = real is faster than stub (we have more headroom)")
print("Positive diff = real is slower than stub (we have less headroom)")

## What to do with these results

1. **If real 0.5B fits (≤9hr projected):**
   - Confirmed: use Qwen2.5-0.5B-Instruct-AWQ as default
   - Download the `real_compute_profile.json` file
   - Update `docs/phase0_compute_profile.md` with the real numbers

2. **If real 0.5B does NOT fit:**
   - We need to optimize: shorter prompts, fewer output tokens, vLLM continuous batching
   - Or fall back to CPU + tiny model (terrible but might work)

3. **If real 1.5B fits (stub said it doesn't):**
   - Upgrade default to 1.5B for better reasoning
   - 1.5B has ~3× the parameter count of 0.5B; should meaningfully improve hypothesis generation

4. **If neither fits:**
   - We're in trouble. Need to revisit architecture (Tier 1 symbolic IG only, very short prompts)

## After measurement

Upload the `real_compute_profile.json` to the AlphaMoo repo and replace the stub-based numbers in `docs/phase0_compute_profile.md`. The stub numbers stay as a historical reference.